# مشروع التنبؤ بارتفاع تلوث الهواء في الرياض

**الهدف:** تدريب نموذج يتوقع ما إذا كان تركيز PM2.5 سيتجاوز 35 ميكروغرام/م³ بعد 24 ساعة. اخترنا هذا الحد للتجربة فقط، وليس بوصفه حد إنذار رسميًا.

**البيانات:** ملف الرياض لعام 2024 من Open-Meteo/CAMS. هذه تقديرات نموذج جوي وليست قراءات محطة رصد محلية.

سننظف البيانات، ونجهز الخصائص، وندرب نموذجين، ثم نقارن النتائج بتوقع بسيط.

## 1. استيراد المكتبات
نستخدم Pandas لتنظيم البيانات وScikit-learn لبناء النماذج وتقييمها.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, recall_score, precision_score,
                             average_precision_score, ConfusionMatrixDisplay)
import joblib

RANDOM_STATE = 42
THRESHOLD = 35.0  # تجريبي، بوحدة µg/m³
HORIZON_HOURS = 24

## 2. قراءة ملف البيانات
نرفع ملف Excel إذا لم يكن موجودًا في Colab، ثم نعرض أول الصفوف.

In [ ]:
file_path = Path('/content/Riyadh_Air_Quality_2024_Real_Data.xlsx')
if not file_path.exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('ارفعي ملف Riyadh_Air_Quality_2024_Real_Data.xlsx')
    file_path = Path(next(iter(uploaded)))

excel = pd.ExcelFile(file_path)
print('Sheets:', excel.sheet_names)
if 'Hourly_Data' not in excel.sheet_names and 'ML_Ready' not in excel.sheet_names:
    raise ValueError('الملف لا يحتوي على Hourly_Data أو ML_Ready')
sheet = 'Hourly_Data' if 'Hourly_Data' in excel.sheet_names else 'ML_Ready'
df = pd.read_excel(file_path, sheet_name=sheet)
print('Rows, columns:', df.shape)
display(df.head())

## 3. تنظيف البيانات
نتأكد من الأعمدة والتواريخ والقيم المفقودة، ونزيل التكرارات. نحتفظ بقراءات التلوث المرتفعة حتى نختبر قدرة النموذج على التعامل معها.

In [ ]:
required = ['datetime','pm2_5','pm10','temperature_2m',
            'relative_humidity_2m','wind_speed_10m']
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f'أعمدة ناقصة: {missing_cols}')

df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
for c in required[1:]:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = (df.dropna(subset=['datetime'])
        .drop_duplicates(subset=['datetime'])
        .sort_values('datetime').reset_index(drop=True))
print('Missing values:')
print(df[required].isna().sum())
print('PM2.5 statistics:')
print(df['pm2_5'].describe())
print('Rows:', len(df))

## 4. تحديد النتيجة التي سنتنبأ بها
نربط كل قراءة بقراءة PM2.5 بعد 24 ساعة فعلية، ثم نضع الفئة 1 عند تجاوز الحد التجريبي و0 في غير ذلك.

In [ ]:
future = df[['datetime', 'pm2_5']].copy()
future['datetime'] = future['datetime'] - pd.Timedelta(hours=HORIZON_HOURS)
future = future.rename(columns={'pm2_5':'pm2_5_next_24h'})
# حذف العمود القديم لتجنب تكراره إن كنا نقرأ ML_Ready
base = df.drop(columns=['pm2_5_next_24h'], errors='ignore')
data = base.merge(future, on='datetime', how='left', validate='one_to_one')
data['High_Pollution'] = np.where(data['pm2_5_next_24h'].notna(),
                                  (data['pm2_5_next_24h'] > THRESHOLD).astype(int), np.nan)
print('Valid target rows:', data['High_Pollution'].notna().sum())
print('Target classes:')
print(data['High_Pollution'].value_counts(dropna=False))
display(data[['datetime','pm2_5','pm2_5_next_24h','High_Pollution']].head())

## 5. استكشاف البيانات
نرسم توزيع تركيز PM2.5 وعدد الحالات في كل فئة.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data['pm2_5'].hist(bins=50, ax=axes[0])
axes[0].set_title('Hourly PM2.5 distribution')
axes[0].set_xlabel('PM2.5 (µg/m³)')
data['High_Pollution'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Target class distribution')
axes[1].set_xlabel('0 = no exceedance, 1 = exceedance')
plt.tight_layout(); plt.show()

## 6. تجهيز الخصائص
نضيف قراءات PM2.5 السابقة ومتوسط آخر ست ساعات. نستخدم المعلومات المتاحة وقت التنبؤ فقط.

In [ ]:
data = data.sort_values('datetime').reset_index(drop=True)
for lag in [1, 6, 24]:
    lookup = data[['datetime','pm2_5']].copy()
    lookup['datetime'] += pd.Timedelta(hours=lag)
    lookup = lookup.rename(columns={'pm2_5': f'pm25_lag_{lag}'})
    data = data.merge(lookup, on='datetime', how='left', validate='one_to_one')

# rolling mean is only valid for consecutive hourly timestamps
is_hourly = data['datetime'].diff().eq(pd.Timedelta(hours=1))
group_id = (~is_hourly).cumsum()
data['pm25_avg_6'] = (data.groupby(group_id)['pm2_5']
                        .transform(lambda s: s.rolling(6, min_periods=6).mean()))
data['hour'] = data['datetime'].dt.hour
data['month'] = data['datetime'].dt.month
features = ['pm2_5','pm10','temperature_2m','relative_humidity_2m',
            'wind_speed_10m','hour','month','pm25_lag_1',
            'pm25_lag_6','pm25_lag_24','pm25_avg_6']
data = data.dropna(subset=features + ['High_Pollution']).copy()
data['High_Pollution'] = data['High_Pollution'].astype(int)
print('ML-ready rows:', len(data))
print(data['High_Pollution'].value_counts())

## 7. تقسيم البيانات
نخصص 60% للتدريب و20% للتحقق و20% للاختبار. نترك فاصلًا زمنيًا قدره 24 ساعة عند حدود الفترات حتى لا تتداخل الأهداف المستقبلية.

In [ ]:
data = data.sort_values('datetime').reset_index(drop=True)
n = len(data)
train_end, val_end = int(n * .60), int(n * .80)
train_start_time = data.iloc[train_end]['datetime']
test_start_time = data.iloc[val_end]['datetime']
train = data.iloc[:train_end].copy()
train = train.loc[train['datetime'] + pd.Timedelta(hours=24) < train_start_time]
val = data.iloc[train_end:val_end].copy()
val = val.loc[val['datetime'] + pd.Timedelta(hours=24) < test_start_time]
test = data.iloc[val_end:].copy()

X_train, y_train = train[features], train['High_Pollution']
X_val, y_val = val[features], val['High_Pollution']
X_test, y_test = test[features], test['High_Pollution']
for name, y_part in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(name, len(y_part), 'class counts:', y_part.value_counts().to_dict())
if y_train.nunique() < 2:
    raise ValueError('بيانات التدريب تحتوي فئة واحدة فقط؛ يجب مراجعة الحد أو الفترة قبل التدريب')

## 8. تدريب النماذج ومقارنتها
نقارن Logistic Regression وRandom Forest بتوقع بسيط يفترض استمرار حالة التلوث الحالية بعد 24 ساعة.

In [ ]:
baseline_val = (X_val['pm2_5'] > THRESHOLD).astype(int)
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=8, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
}
for model in models.values():
    model.fit(X_train, y_train)

val_results = [{'Model':'Baseline',
                'F1':f1_score(y_val,baseline_val,zero_division=0),
                'Recall':recall_score(y_val,baseline_val,zero_division=0),
                'Precision':precision_score(y_val,baseline_val,zero_division=0)}]
for name, model in models.items():
    pred = model.predict(X_val)
    val_results.append({'Model':name,
                        'F1':f1_score(y_val,pred,zero_division=0),
                        'Recall':recall_score(y_val,pred,zero_division=0),
                        'Precision':precision_score(y_val,pred,zero_division=0)})
val_table = pd.DataFrame(val_results).sort_values('F1', ascending=False)
display(val_table.round(3))

## مقارنة النماذج

يعرض الجدول التالي نتائج النماذج الثلاثة على بيانات التحقق (Validation). نختار النموذج حسب F1 للفئة 1، ولا نستخدم بيانات الاختبار لاتخاذ القرار. الأرقام تظهر بعد تشغيل الخلية، ولا نفترض نتائج للنماذج قبل التدريب.


In [ ]:
comparison = val_table[['Model', 'Precision', 'Recall', 'F1']].copy()
display(comparison.round(3).reset_index(drop=True))

ax = comparison.set_index('Model')[['Precision', 'Recall', 'F1']].plot(
    kind='bar', figsize=(9, 4), rot=0
)
ax.set_title('Validation: Model Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 9. اختيار النموذج
نختار النموذج بناءً على نتائج مجموعة التحقق فقط. إذا كان التوقع البسيط أفضل، نذكر ذلك في التقرير.

In [ ]:
best_name = val_table.iloc[0]['Model']
print('Best validation model:', best_name)
if best_name == 'Baseline':
    print('تنبيه: Baseline أفضل من نماذج ML على Validation. هذه نتيجة علمية صحيحة.')
else:
    print('Validation classification report:')
    val_pred = models[best_name].predict(X_val)
    print(classification_report(y_val, val_pred, zero_division=0))
    if y_val.nunique() == 2:
        prob = models[best_name].predict_proba(X_val)[:,1]
        print('Validation Average Precision:', round(average_precision_score(y_val,prob),3))

## 10. الاختبار النهائي
نقيس أداء النموذج المختار على الفترة الأخيرة التي لم نستخدمها في التدريب أو الاختيار.

In [ ]:
baseline_test = (X_test['pm2_5'] > THRESHOLD).astype(int)
test_pred = baseline_test if best_name == 'Baseline' else models[best_name].predict(X_test)
print('Selected model:', best_name)
print(classification_report(y_test, test_pred, zero_division=0))
print('Baseline test F1:', round(f1_score(y_test,baseline_test,zero_division=0),3))
print('Selected test F1:', round(f1_score(y_test,test_pred,zero_division=0),3))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, values_format='d')
plt.title('Final test confusion matrix'); plt.show()
if best_name != 'Baseline' and y_test.nunique() == 2:
    test_prob = models[best_name].predict_proba(X_test)[:,1]
    print('Test Average Precision:', round(average_precision_score(y_test,test_prob),3))

## 11. معرفة الخصائص المؤثرة
نعرض أهمية الخصائص في Random Forest لفهم ما اعتمد عليه النموذج. أهمية الخاصية لا تعني أنها سبب مباشر للتلوث.

In [ ]:
rf = models['Random Forest']
importance = pd.Series(rf.feature_importances_, index=features).sort_values()
importance.plot(kind='barh', figsize=(8,5), title='Random Forest Feature Importance')
plt.tight_layout(); plt.show()

## 12. حفظ النتائج
نحفظ جدول التقييم والنموذج والملفات اللازمة لإضافتها إلى GitHub.

In [ ]:
from pathlib import Path
output_dir = Path('/content/air_quality_outputs')
output_dir.mkdir(exist_ok=True)
val_table.to_csv(output_dir / 'validation_metrics.csv', index=False)
summary = pd.DataFrame([{
    'chosen_model':best_name,
    'threshold_ug_m3':THRESHOLD,
    'test_f1':f1_score(y_test,test_pred,zero_division=0),
    'test_recall':recall_score(y_test,test_pred,zero_division=0),
    'baseline_test_f1':f1_score(y_test,baseline_test,zero_division=0)
}])
summary.to_csv(output_dir / 'test_summary.csv', index=False)
if best_name != 'Baseline':
    joblib.dump({'model':models[best_name], 'features':features,
                 'threshold':THRESHOLD, 'horizon_hours':24},
                output_dir / 'selected_model.joblib')
print('Saved to:', output_dir)
display(summary)

## Conclusion

This project investigated whether air quality and weather measurements could predict if PM2.5 in Riyadh would exceed an experimental threshold of 35 µg/m³ exactly 24 hours later.

Logistic Regression and Random Forest were trained and compared with a persistence baseline. Based on the validation results, the baseline was selected. On the held-out test set, it achieved **86% accuracy**, **91% recall** for high-pollution cases, and an **F1-score of 0.914** for that class. The baseline and selected-model test F1 scores were identical because the selected model was the baseline.

The test set was imbalanced: **1,458 of 1,746 observations** were high-pollution cases. Therefore, accuracy alone does not establish reliable early-warning performance. In this experiment, the machine learning models did not demonstrate an improvement over the simple baseline.

Future work could test additional historical and forecast weather features, evaluate different time periods, and select an alert threshold using an appropriate air-quality standard. The threshold used here is for experimentation, not an official Saudi warning level.

**Interpretation:** The experiment provides a reproducible comparison, but it does not yet validate an operational early-warning system.
